In [1]:
from owlready2 import *

# Load or create an ontology
onto = get_ontology("https://cedric.cnam.fr/~hamdif/ontologies/files/PersonLink.owl").load()

In [2]:
onto

get_ontology("http://cedric.cnam.fr/~hamdif/ontologies/PersonLink.owl#")

In [2]:
# show existing data properties
def get_english_label(prop):
    return prop.label.en
with onto:
    for obj_prop in onto.object_properties():
        # Get the label, if available
        label = get_english_label(obj_prop)  # .label is a list; .first() retrieves the first label
        print(f"Property: {obj_prop.name}, Label: {label}")


def find_property_by_name(onto, prop_name):
    res = []
    for obj_prop in onto.object_properties():
        if len(get_english_label(obj_prop))>0 :
            if get_english_label(obj_prop)[0] == prop_name :
                res.append(obj_prop)
    return res

Property: 1, Label: ['AncestorOf']
Property: ancestorOf, Label: []
Property: 1.1, Label: ['GreatGrandParentOf']
Property: 1.1.1, Label: ['GreatGrandMotherOf']
Property: 1.1.2, Label: ['GreatGrandFatherOf']
Property: 1.2, Label: ['GrandParentOf']
Property: grandparentOf, Label: []
Property: 1.2.1, Label: ['GrandMotherOf']
Property: 1.2.2, Label: ['GrandFatherOf']
Property: 1.3, Label: ['ParentOf']
Property: parentOf, Label: []
Property: 1.3.1, Label: ['MotherOf']
Property: 1.3.1.1, Label: ['LegalMotherOf']
Property: 1.3.1.2, Label: ['BiologicalMotherOf']
Property: 1.3.1.3, Label: ['EggDonorOf']
Property: 1.3.1.4, Label: ['NaturalMotherOf']
Property: 1.3.1.5, Label: ['SurrogateOf']
Property: 1.3.2, Label: ['FatherOf']
Property: 1.3.2.1, Label: ['LegalFatherOf']
Property: 1.3.2.2, Label: ['BiologicalFatherOf']
Property: 1.3.2.3, Label: ['DonorFatherOf']
Property: 1.3.2.4, Label: ['NaturalFatherOf']
Property: 10, Label: ['NieceOf']
Property: 10.1, Label: []
Property: 10.2, Label: []
Proper

In [4]:
find_property_by_name(onto, "FatherOf")
#1.3.1
#1.3.2

[PersonLink.1.3.2]

In [41]:
with onto:
    # Define the SWRL rule:
    # Body of the rule: If x is related to y via 3.1.1 and y is related to z, and z is a "Homme"
    # Rule 1: Body and Head
    rule1 = Imp()
    rule1.set_as_rule("""
    PersonLink.1.3.1(?x, ?y),  PersonLink.1.3.1(?y, ?z), Homme(?z) -> Male(?z, ?x)
    """)

ValueError: Cannot find entity 'PersonLink.1.3.1'!

In [7]:
from rdflib import Graph, Namespace, RDF, URIRef
from rdflib.namespace import OWL

def create_swrl_rule():
    # Define namespaces
    SWRL = Namespace("http://www.w3.org/2003/11/swrl#")
    RDF_NS = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")
    EX = Namespace("urn:swrl#")
    
    # Create RDF graph
    g = Graph()
    g.bind("swrl", SWRL)
    g.bind("rdf", RDF_NS)
    g.bind("ex", EX)
    
    # Define SWRL rule
    rule = URIRef(EX["Rule1"])
    head = URIRef(EX["head"])
    body = URIRef(EX["body"])
    
    g.add((rule, RDF.type, SWRL["Imp"]))
    g.add((rule, SWRL["head"], head))
    g.add((rule, SWRL["body"], body))
    
    # Define Head AtomList
    head_list = URIRef(EX["headList"])
    g.add((head, SWRL["AtomList"], head_list))
    g.add((head_list, RDF["rest"], RDF_NS["nil"]))
    
    head_first = URIRef(EX["headFirst"])
    g.add((head_list, RDF["first"], head_first))
    g.add((head_first, RDF.type, SWRL["IndividualPropertyAtom"]))
    g.add((head_first, SWRL["propertyPredicate"], URIRef("#102")))
    g.add((head_first, SWRL["argument1"], EX["x"]))
    g.add((head_first, SWRL["argument2"], EX["y"]))
    
    # Define Body AtomList
    body_list = URIRef(EX["bodyList"])
    g.add((body, SWRL["AtomList"], body_list))
    
    body_first = URIRef(EX["bodyFirst"])
    g.add((body_list, RDF["first"], body_first))
    g.add((body_first, RDF.type, SWRL["IndividualPropertyAtom"]))
    g.add((body_first, SWRL["propertyPredicate"], URIRef("#1.3.2")))
    g.add((body_first, SWRL["argument1"], EX["x"]))
    g.add((body_first, SWRL["argument2"], EX["y"]))
    
    body_rest = URIRef(EX["bodyRest"])
    g.add((body_list, RDF["rest"], body_rest))
    g.add((body_rest, SWRL["AtomList"], URIRef(EX["bodyNext"])))
    
    body_next = URIRef(EX["bodyNext"])
    g.add((body_next, RDF["rest"], RDF_NS["nil"]))
    
    body_next_first = URIRef(EX["bodyNextFirst"])
    g.add((body_next, RDF["first"], body_next_first))
    g.add((body_next_first, RDF.type, SWRL["IndividualPropertyAtom"]))
    g.add((body_next_first, SWRL["propertyPredicate"], URIRef("#5.2")))
    g.add((body_next_first, SWRL["argument1"], EX["x"]))
    g.add((body_next_first, SWRL["argument2"], EX["y"]))
    
    # Properly reference head and body descriptions
    g.add((head, SWRL["AtomList"], head_list))
    g.add((body, SWRL["AtomList"], body_list))
    g.add((body_rest, SWRL["AtomList"], body_next))
    
    return g.serialize(format="pretty-xml")

# Print the OWL with the SWRL rule
print(create_swrl_rule())

<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF
  xmlns:swrl="http://www.w3.org/2003/11/swrl#"
  xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
>
  <swrl:Imp rdf:about="urn:swrl#Rule1">
    <swrl:head>
      <rdf:Description rdf:about="urn:swrl#head">
        <swrl:AtomList rdf:parseType="Collection">
          <rdf:Description rdf:about="urn:swrl#headFirst"/>
        </swrl:AtomList>
      </rdf:Description>
    </swrl:head>
    <swrl:body>
      <rdf:Description rdf:about="urn:swrl#body">
        <swrl:AtomList rdf:parseType="Collection">
          <rdf:Description rdf:about="urn:swrl#bodyFirst"/>
        </swrl:AtomList>
      </rdf:Description>
    </swrl:body>
  </swrl:Imp>
  <swrl:IndividualPropertyAtom rdf:about="urn:swrl#headFirst">
    <swrl:propertyPredicate rdf:resource="#102"/>
    <swrl:argument1 rdf:resource="urn:swrl#x"/>
    <swrl:argument2 rdf:resource="urn:swrl#y"/>
  </swrl:IndividualPropertyAtom>
  <rdf:Description rdf:about="urn:swrl#bodyRest">
    <swrl:A

c:\Python312\Lib\site-packages\rdflib\plugins\serializers\rdfxml.py:280: UserWarning: Assertions on rdflib.term.URIRef('urn:swrl#headList') other than RDF.first and RDF.rest are ignored ... including RDF.List
  self.predicate(predicate, object, depth + 1)
c:\Python312\Lib\site-packages\rdflib\plugins\serializers\rdfxml.py:280: UserWarning: Assertions on rdflib.term.URIRef('urn:swrl#bodyList') other than RDF.first and RDF.rest are ignored ... including RDF.List
  self.predicate(predicate, object, depth + 1)
c:\Python312\Lib\site-packages\rdflib\plugins\serializers\rdfxml.py:280: UserWarning: Assertions on rdflib.term.URIRef('urn:swrl#bodyNext') other than RDF.first and RDF.rest are ignored ... including RDF.List
  self.predicate(predicate, object, depth + 1)


In [ ]:
<swrl:Imp>
  <swrl:head>
    <swrl:AtomList>
      <rdf:rest rdf:resource="http://www.w3.org/1999/02/22-rdf-syntax-ns#nil"/>
      <rdf:first>
        <swrl:IndividualPropertyAtom>
          <swrl:propertyPredicate rdf:resource="#102"/>
          <swrl:argument1 rdf:resource="urn:swrl#x"/>
          <swrl:argument2 rdf:resource="urn:swrl#y"/>
        </swrl:IndividualPropertyAtom>
      </rdf:first>
    </swrl:AtomList>
  </swrl:head>
  <swrl:body>
    <swrl:AtomList>
      <rdf:first>
        <swrl:IndividualPropertyAtom>
          <swrl:propertyPredicate rdf:resource="#1.3.2"/>
          <swrl:argument1 rdf:resource="urn:swrl#x"/>
          <swrl:argument2 rdf:resource="urn:swrl#y"/>
        </swrl:IndividualPropertyAtom>
      </rdf:first>
      <rdf:rest>
        <swrl:AtomList>
          <rdf:rest rdf:resource="http://www.w3.org/1999/02/22-rdf-syntax-ns#nil"/>
          <rdf:first>
            <swrl:IndividualPropertyAtom>
              <swrl:propertyPredicate rdf:resource="#5.2"/>
              <swrl:argument1 rdf:resource="urn:swrl#x"/>
              <swrl:argument2 rdf:resource="urn:swrl#y"/>
            </swrl:IndividualPropertyAtom>
          </rdf:first>
        </swrl:AtomList>
      </rdf:rest>
    </swrl:AtomList>
  </swrl:body>
</swrl:Imp>

In [26]:
import xml.etree.ElementTree as ET
# Create root element
root = ET.Element("swrl:Imp")

#BULID right sight of the implication 
head = ET.SubElement(root, "swrl:head")
AtomList = ET.SubElement(head,"swrl:AtomList")
ET.SubElement(AtomList,"rdf:rest",xmlns_rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#nil")
first = ET.SubElement(AtomList,"rdf:first")
IndividualPropertyAtom = ET.SubElement(first,"swrl:IndividualPropertyAtom")
ET.SubElement(IndividualPropertyAtom,"swrl:propertyPredicate",{"rdf:resource": "#102"})
ET.SubElement(IndividualPropertyAtom,"swrl:argument1",{"rdf:resource": "urn:swrl#x"})
ET.SubElement(IndividualPropertyAtom,"swrl:argument2",{"rdf:resource": "urn:swrl#y"})


# Left side
body = ET.SubElement(root, "swrl:body")
AtomList = ET.SubElement(body,"swrl:AtomList")
# First Argumnet on the left side 
first = ET.SubElement(AtomList,"rdf:first")
IndividualPropertyAtom = ET.SubElement(first,"swrl:IndividualPropertyAtom")
ET.SubElement(IndividualPropertyAtom,"swrl:propertyPredicate",{"rdf:resource": "#1.3.2"})
ET.SubElement(IndividualPropertyAtom,"swrl:argument1",{"rdf:resource": "urn:swrl#x"})
ET.SubElement(IndividualPropertyAtom,"swrl:argument2",{"rdf:resource": "urn:swrl#y"})

# Next arguments on the left side 
rest = ET.SubElement(AtomList,"rdf:rest")
InsideAtomList = ET.SubElement(rest,"swrl:AtomList")
insideRest = ET.SubElement(InsideAtomList,"rdf:rest",xmlns_rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#nil")
insdiefirst = ET.SubElement(InsideAtomList,"rdf:first")
IndividualPropertyAtom = ET.SubElement(insdiefirst,"swrl:IndividualPropertyAtom")
ET.SubElement(IndividualPropertyAtom,"swrl:propertyPredicate",{"rdf:resource": "#5.2"})
ET.SubElement(IndividualPropertyAtom,"swrl:argument1",{"rdf:resource": "urn:swrl#x"})
ET.SubElement(IndividualPropertyAtom,"swrl:argument2",{"rdf:resource": "urn:swrl#y"})

# Convert to a string and print
tree = ET.ElementTree(root)
xml_string = ET.tostring(root, encoding="utf-8").decode("utf-8")
print(xml_string)

<swrl:Imp><swrl:head><swrl:AtomList><rdf:rest xmlns_rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#nil" /><rdf:first><swrl:IndividualPropertyAtom><swrl:propertyPredicate rdf:resource="#102" /><swrl:argument1 rdf:resource="urn:swrl#x" /><swrl:argument2 rdf:resource="urn:swrl#y" /></swrl:IndividualPropertyAtom></rdf:first></swrl:AtomList></swrl:head><swrl:body><swrl:AtomList><rdf:first><swrl:IndividualPropertyAtom><swrl:propertyPredicate rdf:resource="#1.3.2" /><swrl:argument1 rdf:resource="urn:swrl#x" /><swrl:argument2 rdf:resource="urn:swrl#y" /></swrl:IndividualPropertyAtom></rdf:first><rdf:rest><swrl:AtomList><rdf:rest xmlns_rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#nil" /><rdf:first><swrl:IndividualPropertyAtom><swrl:propertyPredicate rdf:resource="#5.2" /><swrl:argument1 rdf:resource="urn:swrl#x" /><swrl:argument2 rdf:resource="urn:swrl#y" /></swrl:IndividualPropertyAtom></rdf:first></swrl:AtomList></rdf:rest></swrl:AtomList></swrl:body></swrl:Imp>


In [32]:
def remove_last_n_lines(file_path, n):
    try:
        # Open file with utf-8 encoding to handle non-ASCII characters
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
        
        # Remove the last 'n' lines
        new_lines = lines[:-n]  # Slicing removes the last n lines
        
        # Write the updated content back to the file
        with open(file_path, 'w', encoding='utf-8') as file:
            file.writelines(new_lines)
    
    except UnicodeDecodeError as e:
        print(f"Error reading the file: {e}")
        # You can try using 'ISO-8859-1' if utf-8 doesn't work
        with open(file_path, 'r', encoding='ISO-8859-1') as file:
            lines = file.readlines()
        
        # Remove the last 'n' lines
        new_lines = lines[:-n]  # Slicing removes the last n lines
        
        # Write the updated content back to the file
        with open(file_path, 'w', encoding='ISO-8859-1') as file:
            file.writelines(new_lines)

# Example usage
remove_last_n_lines('example.owl', 3)  # Removes the last 3 lines


In [33]:
def append_to_file(file_path, text_to_append):
    with open(file_path, 'a', encoding='utf-8') as file:
        file.write(text_to_append + '\n')  # Adding a newline for each appended text

# Example usage
append_to_file('example.owl', xml_string)
append_to_file("example.owl","</rdf:RDF>")

In [23]:
from lxml import etree

# Define namespaces with prefixes
nsmap = {
    "swrl": "http://www.w3.org/2003/11/swrl#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
}

# Create root element with the swrl prefix
root = etree.Element("swrl:Imp", nsmap=nsmap)

# Build right side of the implication (head)
head = etree.SubElement(root, "swrl:head")
AtomList = etree.SubElement(head, "swrl:AtomList")
etree.SubElement(AtomList, "rdf:rest", attrib={"rdf:resource": "http://www.w3.org/1999/02/22-rdf-syntax-ns#nil"})
first = etree.SubElement(AtomList, "rdf:first")
IndividualPropertyAtom = etree.SubElement(first, "swrl:IndividualPropertyAtom")
etree.SubElement(IndividualPropertyAtom, "swrl:propertyPredicate", attrib={"rdf:resource": "#102"})
etree.SubElement(IndividualPropertyAtom, "swrl:argument1", attrib={"rdf:resource": "urn:swrl#x"})
etree.SubElement(IndividualPropertyAtom, "swrl:argument2", attrib={"rdf:resource": "urn:swrl#y"})

# Left side of the implication (body)
body = etree.SubElement(root, "swrl:body")
AtomList = etree.SubElement(body, "swrl:AtomList")
first = etree.SubElement(AtomList, "rdf:first")
IndividualPropertyAtom = etree.SubElement(first, "swrl:IndividualPropertyAtom")
etree.SubElement(IndividualPropertyAtom, "swrl:propertyPredicate", attrib={"rdf:resource": "#1.3.2"})
etree.SubElement(IndividualPropertyAtom, "swrl:argument1", attrib={"rdf:resource": "urn:swrl#x"})
etree.SubElement(IndividualPropertyAtom, "swrl:argument2", attrib={"rdf:resource": "urn:swrl#y"})

# Additional atoms on the left side
rest = etree.SubElement(AtomList, "rdf:rest")
InsideAtomList = etree.SubElement(rest, "swrl:AtomList")
etree.SubElement(InsideAtomList, "rdf:rest", attrib={"rdf:resource": "http://www.w3.org/1999/02/22-rdf-syntax-ns#nil"})
insidefirst = etree.SubElement(InsideAtomList, "rdf:first")
IndividualPropertyAtom = etree.SubElement(insidefirst, "swrl:IndividualPropertyAtom")
etree.SubElement(IndividualPropertyAtom, "swrl:propertyPredicate", attrib={"rdf:resource": "#5.2"})
etree.SubElement(IndividualPropertyAtom, "swrl:argument1", attrib={"rdf:resource": "urn:swrl#x"})
etree.SubElement(IndividualPropertyAtom, "swrl:argument2", attrib={"rdf:resource": "urn:swrl#y"})

# Convert to a string and print it
xml_string = etree.tostring(root, encoding="utf-8", pretty_print=True).decode("utf-8")

# Print the XML string
print(xml_string)


ValueError: Invalid tag name 'swrl:Imp'